Load Data into SQL

In [42]:
import pandas as pd
import sqlite3

Load Cleaned Data

In [43]:
users = pd.read_csv("../data/processed/users_cleaned.csv")
transactions = pd.read_csv("../data/processed/transactions_cleaned.csv")
campaigns = pd.read_csv("../data/processed/campaigns_cleaned.csv")

Create Database

In [44]:
conn = sqlite3.connect("../data/fintech.db")

Load Tables into SQL

In [45]:
users.to_sql("users", conn, if_exists="replace", index=False)
transactions.to_sql("transactions", conn, if_exists="replace", index=False)
campaigns.to_sql("campaigns", conn, if_exists="replace", index=False)

10

In [46]:
pd.read_sql("SELECT * FROM users LIMIT 5", conn)

,user_id,signup_date,country,age,gender,acquisition_channel,campaign_id,kyc_completed,kyc_date,first_transaction_date,total_transactions,total_amount,avg_transaction_value,is_converted,is_active_user,kyc_done_flag,transaction_done_flag,cohort_month
0,USR_1,2023-04-13,India,25,Male,Referral,CMP_4,1,2023-04-16,2023-05-23,10.0,44526.59,4452.659000,1,1,1,1,2023-04
1,USR_2,2023-08-03,UK,57,Female,Google,CMP_8,1,2023-08-05,2023-09-13,5.0,29532.43,5906.486000,1,1,1,1,2023-08
2,USR_3,2023-12-10,Canada,38,Male,Facebook,CMP_6,1,2023-12-11,2023-12-23,48.0,226313.42,4714.862917,1,1,1,1,2023-12
3,USR_4,2023-08-24,Canada,59,Female,Google,CMP_9,0,NaN,NaN,0.0,0.00,0.000000,0,0,0,0,2023-08
4,USR_5,2023-09-28,UK,20,Male,Google,CMP_3,1,2023-09-29,NaN,0.0,0.00,0.000000,0,0,1,0,2023-09


ANALYSIS 1 — Funnel Conversion

Funnel Query

In [47]:
query = """
SELECT
    COUNT(*) AS total_users,
    
    SUM(kyc_done_flag) AS kyc_completed_users,
    
    SUM(transaction_done_flag) AS converted_users,
    
    ROUND(100.0 * SUM(kyc_done_flag) / COUNT(*), 2) AS kyc_conversion_rate,
    
    ROUND(100.0 * SUM(transaction_done_flag) / COUNT(*), 2) AS overall_conversion_rate

FROM users
"""

funnel_df = pd.read_sql(query, conn)
funnel_df

,total_users,kyc_completed_users,converted_users,kyc_conversion_rate,overall_conversion_rate
0,5000,3983,3681,79.66,73.62


ANALYSIS 2 — Campaign Performance

In [48]:
query = """
SELECT
    c.campaign_name,
    COUNT(u.user_id) AS total_users,
    SUM(u.transaction_done_flag) AS converted_users,
    
    ROUND(100.0 * SUM(u.transaction_done_flag) / COUNT(u.user_id), 2) AS conversion_rate

FROM users u
JOIN campaigns c
ON u.campaign_id = c.campaign_id

GROUP BY c.campaign_name
ORDER BY conversion_rate DESC
"""

campaign_perf = pd.read_sql(query, conn)
campaign_perf

,campaign_name,total_users,converted_users,conversion_rate
0,Campaign_1,531,403,75.89
1,Campaign_7,498,377,75.70
2,Campaign_8,468,350,74.79
3,Campaign_9,474,354,74.68
4,Campaign_4,484,360,74.38
5,Campaign_5,490,360,73.47
6,Campaign_3,548,400,72.99
7,Campaign_10,538,389,72.30
8,Campaign_2,475,340,71.58
9,Campaign_6,494,348,70.45


ANALYSIS 3 — User Activity Segmentation

In [49]:
query = """
SELECT
    CASE
        WHEN total_transactions = 0 THEN 'Inactive'
        WHEN total_transactions BETWEEN 1 AND 5 THEN 'Low Activity'
        WHEN total_transactions BETWEEN 6 AND 20 THEN 'Medium Activity'
        ELSE 'High Activity'
    END AS user_segment,
    
    COUNT(*) AS user_count

FROM users

GROUP BY user_segment
ORDER BY user_count DESC
"""

segment_df = pd.read_sql(query, conn)
segment_df

,user_segment,user_count
0,Medium Activity,2042
1,Inactive,1319
2,Low Activity,1249
3,High Activity,390


Retention Analysis (Cohort Analysis)

Get required data

In [50]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/fintech.db")

query = """
SELECT 
    u.user_id,
    u.signup_date,
    t.transaction_date
FROM users u
JOIN transactions t
ON u.user_id = t.user_id
WHERE t.status = 'success'
"""

df = pd.read_sql(query, conn)

In [51]:
df["signup_date"] = pd.to_datetime(df["signup_date"])
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

Create Cohort Columns

In [52]:
df["cohort_month"] = df["signup_date"].dt.to_period("M")
df["transaction_month"] = df["transaction_date"].dt.to_period("M")

Cohort Index


In [53]:
df["cohort_index"] = (
    (df["transaction_month"].dt.year - df["cohort_month"].dt.year) * 12 +
    (df["transaction_month"].dt.month - df["cohort_month"].dt.month)
)

In [54]:
#Remove invalid rows
df = df[df["cohort_index"] >= 0]

In [55]:
#Remove duplicate users
df_unique = df.drop_duplicates(subset=["user_id", "cohort_month", "cohort_index"])

In [56]:
df_unique.duplicated(subset=["user_id", "cohort_month", "cohort_index"]).sum()

np.int64(0)

In [57]:
#Cohert table
cohort_data = df_unique.groupby(
    ["cohort_month", "cohort_index"]
)["user_id"].nunique().reset_index()

In [58]:
#Pivot
cohort_pivot = cohort_data.pivot_table(
    index="cohort_month",
    columns="cohort_index",
    values="user_id"
)

Retention %

In [59]:
users = pd.read_sql("SELECT user_id, signup_date FROM users", conn)

users["signup_date"] = pd.to_datetime(users["signup_date"])

cohort_size = users.groupby(
    users["signup_date"].dt.to_period("M")
)["user_id"].nunique()

cohort_size.index.name = "cohort_month"

In [60]:
retention = cohort_pivot.divide(cohort_size, axis=0)

retention = retention.round(3)
retention

cohort_index,0,1,2,3,4,5,6
cohort_month,,,,,,,
2023-01,0.386,0.566,0.576,0.568,0.568,0.547,0.374
2023-02,0.319,0.610,0.560,0.558,0.558,0.545,0.327
2023-03,0.349,0.549,0.565,0.574,0.535,0.586,0.307
2023-04,0.355,0.587,0.550,0.572,0.570,0.584,0.311
2023-05,0.371,0.514,0.545,0.568,0.545,0.538,0.322
2023-06,0.371,0.604,0.576,0.573,0.576,0.597,0.298
2023-07,0.351,0.577,0.552,0.566,0.568,0.575,0.310
2023-08,0.439,0.584,0.600,0.602,0.597,0.613,0.299
2023-09,0.364,0.538,0.567,0.564,0.521,0.531,0.300


In [61]:
retention[0] = 1.0

In [62]:
retention = retention.sort_index(axis=1)

In [63]:
retention = retention.round(3)
retention

cohort_index,0,1,2,3,4,5,6
cohort_month,,,,,,,
2023-01,1.0,0.566,0.576,0.568,0.568,0.547,0.374
2023-02,1.0,0.610,0.560,0.558,0.558,0.545,0.327
2023-03,1.0,0.549,0.565,0.574,0.535,0.586,0.307
2023-04,1.0,0.587,0.550,0.572,0.570,0.584,0.311
2023-05,1.0,0.514,0.545,0.568,0.545,0.538,0.322
2023-06,1.0,0.604,0.576,0.573,0.576,0.597,0.298
2023-07,1.0,0.577,0.552,0.566,0.568,0.575,0.310
2023-08,1.0,0.584,0.600,0.602,0.597,0.613,0.299
2023-09,1.0,0.538,0.567,0.564,0.521,0.531,0.300


In [64]:
retention.max().max()

np.float64(1.0)

In [65]:
retention.to_csv("../data/processed/retention.csv")

ADVANCED CAMPAIGN ANALYSIS

In [66]:
query = """
SELECT
    c.campaign_name,
    COUNT(DISTINCT u.user_id) AS users,
    SUM(u.total_amount) AS revenue,
    c.cost,
    ROUND(SUM(u.total_amount) / COUNT(DISTINCT u.user_id), 2) AS arpu,
    ROUND((SUM(u.total_amount) - c.cost) / c.cost, 2) AS roi
FROM users u
JOIN campaigns c
ON u.campaign_id = c.campaign_id
GROUP BY c.campaign_name, c.cost
ORDER BY roi DESC
"""

campaign_advanced = pd.read_sql(query, conn)
campaign_advanced

,campaign_name,users,revenue,cost,arpu,roi
0,Campaign_10,538,26583102.70,28976.99,49410.97,916.39
1,Campaign_3,548,27564664.76,30484.32,50300.48,903.22
2,Campaign_4,484,23945109.00,30298.24,49473.37,789.31
3,Campaign_8,468,24698129.87,54606.78,52773.78,451.29
4,Campaign_5,490,23136878.03,53812.12,47218.12,428.96
5,Campaign_1,531,28096395.10,76341.67,52912.23,367.03
6,Campaign_2,475,21228151.06,68436.70,44690.84,309.19
7,Campaign_6,494,23587741.79,78916.57,47748.47,297.89
8,Campaign_7,498,24019331.70,90570.94,48231.59,264.20
9,Campaign_9,474,23007568.14,97215.85,48539.17,235.66


CUSTOMER SEGMENTATION (RFM)

In [67]:
query = """
WITH max_date AS (
    SELECT MAX(transaction_date) AS max_txn_date
    FROM transactions
)

SELECT
    t.user_id,
    
    JULIANDAY(m.max_txn_date) - JULIANDAY(MAX(t.transaction_date)) AS recency,
    
    COUNT(t.transaction_id) AS frequency,
    
    SUM(t.amount) AS monetary

FROM transactions t
CROSS JOIN max_date m
WHERE t.status = 'success'

GROUP BY t.user_id
"""

rfm = pd.read_sql(query, conn)
rfm.head()

,user_id,recency,frequency,monetary
0,USR_1,274.0,10,44526.59
1,USR_10,16.0,19,73499.95
2,USR_100,171.0,4,28960.47
3,USR_1000,258.0,10,55789.78
4,USR_1001,418.0,1,158.98


CREATE SEGMENTS

In [68]:
def segment_user(row):
    if row['monetary'] > 100000 and row['frequency'] > 20:
        return 'High Value'
    elif row['frequency'] > 10:
        return 'Loyal'
    elif row['recency'] > 60:
        return 'At Risk'
    else:
        return 'Low Value'

rfm['segment'] = rfm.apply(segment_user, axis=1)

rfm['segment'].value_counts()

segment
At Risk       2239
Loyal          831
High Value     390
Low Value      221
Name: count, dtype: int64

MERGE WITH USERS

In [69]:
users_segmented = users.merge(rfm, on='user_id', how='left')

# Fill missing values (important)
users_segmented['frequency'] = users_segmented['frequency'].fillna(0)
users_segmented['monetary'] = users_segmented['monetary'].fillna(0)

users_segmented['segment'] = users_segmented['segment'].fillna('No Activity')
users_segmented['recency'] = users_segmented['recency'].fillna(999)

users_segmented.head()

,user_id,signup_date,recency,frequency,monetary,segment
0,USR_1,2023-04-13,274.0,10.0,44526.59,At Risk
1,USR_2,2023-08-03,186.0,5.0,29532.43,At Risk
2,USR_3,2023-12-10,21.0,48.0,226313.42,High Value
3,USR_4,2023-08-24,999.0,0.0,0.00,No Activity
4,USR_5,2023-09-28,999.0,0.0,0.00,No Activity


In [70]:
users_segmented.to_csv("../data/processed/users_segmented.csv", index=False)
campaign_advanced.to_csv("../data/processed/campaign_advanced.csv", index=False)